# **<font color='#6edce9'>ETL: Consolidación Histórica y Homologación de Dotación de RRHH (2019-2025)</font>**
### **Objetivo:** Construir el pipeline de mapeo y limpieza para unificar los datasets anuales de RRHH y migrar la base de datos consolidada a PostgreSQL.

### **Criterios Clave del Proyecto:**
* **Limpieza Estricta:** Eliminación de espacios en blanco (`.strip()`) en nombres de columnas para evitar fallas de indexación.
* **Preservación de Identificadores:** Carga de campos críticos (`RENAES`, `id_cargo`) estrictamente como texto (`str`) para no perder ceros a la izquierda.
* **Optimización de Memoria:** Uso de lecturas parciales (`nrows=0`) durante la fase de mapeo estructural.
* **Destino DB:** Preparación de variables aptas para la sintaxis nativa de PostgreSQL.


## **<font color='#eda985'> 1. Configuración y Funciones Globales </font>**

In [14]:
import pandas as pd
from sqlalchemy import create_engine

# =============================================================================
# A. CONFIGURACIÓN ESTRUCTURAL (DICCIONARIO DE ARCHIVOS)
# =============================================================================
# A.1. Diccionario de rutas y fila de header por año
archivos = {
    2019: ('../data/raw/BASE_DIC_2019.xlsx', 5),
    2020: ('../data/raw/BASE_DIC_2020.xlsx', 0),
    2021: ('../data/raw/BASE_DIC_2021.xlsx', 0),
    2022: ('../data/raw/BASE_DIC_2022.xlsx', 0),
    2023: ('../data/raw/BASE_DIC_2023.xlsx', 0),
    2024: ('../data/raw/BASE_DIC_2024.xlsx', 1),
    2025: ('../data/raw/BASE_DIC_2025.xlsx', 0)
}

In [15]:
# =============================================================================
# B. FUNCIONES GLOBALES
# =============================================================================

# B.1. FUNCIÓN PARA LECTURA RÁPIDA DE COLUMNAS (FASE DE MAPEO)

def obtener_columnas_iniciales(ruta, header_row):
    """Lee solo la cabecera del Excel para no saturar memoria RAM."""
    df_temp = pd.read_excel(ruta, header=header_row, nrows=0)
    # Limpia espacios y descarta columnas vacías/automáticas
    return [str(c).strip() for c in df_temp.columns if c and not str(c).startswith('Unnamed:')]


In [16]:
# B.2. FUNCIÓN DE COMPARACIÓN DE METADATOS (AÑO VS AÑO)

def comparar_columnas_anios(columnas_anio_a, columnas_anio_b, anio_a,  anio_b):
    """Compara estructuralmente las columnas de dos años para detectar discrepancias."""
    cols_a = set(columnas_anio_a)
    cols_b = set(columnas_anio_b)

    solo_en_a = cols_a - cols_b
    solo_en_b = cols_b - cols_a
    en_ambos = cols_a & cols_b

    print(f'---- Coinciden exactamente ({len(en_ambos)}) ----')
    for c in sorted(en_ambos): print(' ', c)
    print(f'---- Solo en {anio_a} ({len(solo_en_a)}) ----')
    for c in sorted(solo_en_a): print(' ', c)
    print(f'---- Solo en {anio_b} ({len(solo_en_b)}) ----')
    for c in sorted(solo_en_b): print(' ', c)

In [17]:
# B.3. FUNCION DE COMPARA VALORES UNICOS (CAMPO VS CAMPO)

def comparar_valores(df_a, col_a, nombre_a, df_b, col_b, nombre_b):
    """Compara el contenido real y valores únicos de dos columnas sospechosas."""
    vals_a = sorted(df_a[col_a].dropna().unique().tolist())
    vals_b = sorted(df_b[col_b].dropna().unique().tolist())

    print(f"{nombre_a} ({col_a}): {len(vals_a)} valores únicos")
    print(f"{nombre_b} ({col_b}): {len(vals_b)} valores únicos")
    print(f"¿Son el mismo conjunto? {set(vals_a) == set(vals_b)}")
    

    return pd.DataFrame({
        f'{nombre_a} ({col_a})': pd.Series(vals_a),
        f'{nombre_b} ({col_b})': pd.Series(vals_b)
    })

In [18]:
# B.4. FUNCIÓN PARA CARGA REAL Y LIMPIEZA PROFUNDA DE DATAFRAMES

def cargar_dataframe_limpio(anio, config_archivos):
    """Carga el DataFrame completo forzando IDs como texto y limpiando columnas."""
    ruta, header_row = config_archivos[anio]
    
    # Crucial para PostgreSQL: Evita que RENAES o IDs pierdan ceros a la izquierda
    dtypes_dict = {
        'REANES FINAL': str, 'RENAES': str, 
        'id_cargo': str, 'id_cargo_recod': str,
        'CODCARGO': str
    }
    
    df = pd.read_excel(ruta, header=header_row, dtype=dtypes_dict)
    df.columns = df.columns.str.strip() # Limpieza de espacios en blanco
    
    # Filtrar columnas Unnamed reales
    columnas_validas = [c for c in df.columns if c and not str(c).startswith('Unnamed:')]
    return df[columnas_validas]

In [19]:
# B.5. FUNCION PARA CONVERTIR FECHA DE NACIMIENTO A TIPO DATO DATETIME

def convertir_fecha_nacimiento(df, columna_original='fecha_nacimiento', formatos = ('%d/%m/%Y', '%Y-%m-%d', '%d-%m-%Y') ):
    """Convierte la columna de fecha de nacimiento a datetime manejando formatos mixtos."""
    nueva_columna = f'{columna_original}_dt'
    df[nueva_columna] = pd.NaT # Prepara la columna con un tipo de dato temporal (datetime), lista para recibir fechas en el futuro sin generar errores de compatibilidad.

    for formato in formatos:
        pendientes = df[columna_original].notna() & df[nueva_columna].isna() #Se identifica los que tienen datos pero en la conversion son nulos
        df.loc[pendientes, nueva_columna] = pd.to_datetime(
            df.loc[pendientes, columna_original], format=formato, errors='coerce'
            )

    nulos_originales = df[columna_original].isna().sum()
    nulos_despues = df[nueva_columna].isna().sum()
    no_convertidos = nulos_despues - nulos_originales

    print(f'Nulos originales: {nulos_originales}')
    print(f'Nulos despues de convertir: {nulos_despues}')
    print(f'No se pueden convertir (formato distinto): {no_convertidos}')

    return df

In [20]:
# B.6. FUNCION PARA CALCULAR EDAD DESDE LA FECHA DE NACIMIENTO

def calcular_edad(fecha_nacimiento, anio_corte):    
    """Calcula la edad entera en años al 31 de diciembre del año de corte."""
    fecha_corte = pd.Timestamp(year=anio_corte, month=12, day=31)
    edad = (fecha_corte - fecha_nacimiento).dt.days// 365
    return edad

In [21]:
# B.7. FUNCIÓN PARA PROCESAR Y VALIDAR COLUMNAS FINALES (POST-MAPEO)

def obtener_columnas_finales(columnas_del_anio, anio, mapeo_columnas, columnas_descartadas):
    """Devuelve los nombres definitivos tras aplicar reglas de negocio y descartes."""
    descartadas = set(columnas_descartadas.get(anio, {}).keys())
    mapeo = mapeo_columnas.get(anio, {})

    mantenidas = set(columnas_del_anio) - descartadas
    finales = {mapeo.get(col, col) for col in mantenidas}
    return sorted(finales)

In [22]:
# B.8. FUNCIÓN DE CONSOLIDACIÓN Y ESTANDARIZACIÓN (FASE ETL)

def consolidar_año(df, anio, mapeo_columnas, columnas_descartadas):
    """Aplica descartes, renombra columnas y añade la etiqueta del año correspondiente."""
    # Clonar para no alterar el DataFrame original en memoria
    df_proc = df.copy()
    
    # 1. Descartar no deseadas (filtrando solo las que existan en el DF actual)
    descartar = [c for c in columnas_descartadas.get(anio, {}).keys() if c in df_proc.columns]
    df_proc.drop(columns=descartar, inplace=True)
    
    # 2. Renombrar según el diccionario canónico
    mapeo = mapeo_columnas.get(anio, {})
    df_proc.rename(columns=mapeo, inplace=True)
    
    # 3. Formatear nombres a minúsculas y snake_case para compatibilidad PostgreSQL
    df_proc.columns = df_proc.columns.str.lower().str.replace(' ', '_').str.replace('+', 'mas')
    
    # 4. Trazabilidad: Inyectar el año del registro
    df_proc['anio_registro'] = anio
    return df_proc

In [23]:
# B.9. FUNCIÓN DE INGESTA A BASE DE DATOS (POSTGRESQL)

def exportar_a_postgresql(df_consolidado, tabla_destino, usuario, password, host, puerto, bd):
    """Realiza la carga masiva indexada del dataset unificado a PostgreSQL."""
    str_conexion = f'postgresql://{usuario}:{password}@{host}:{puerto}/{bd}'
    engine = create_engine(str_conexion)
    
    print(f"Iniciando volcado masivo en la tabla '{tabla_destino}'...")
    # chunksize fragmenta la carga para evitar colapsar la memoria de la BD
    df_consolidado.to_sql(name=tabla_destino, con=engine, if_exists='replace', index=False, chunksize=5000)
    print("¡Conexión y carga masiva finalizada con éxito en PostgreSQL!")

## **<font color='#eda985'>  2. Fase de Exploración y Mapeo Extremo: 2019 vs 2025 </font>**

In [28]:
# =============================================================================
# 2. FASE DE EXPLORACIÓN Y MAPEO EXTREMO: 2019 VS 2025
# =============================================================================

# 2.1 Extracción e inspección rápida de nombres de columnas

columnas_2019 = obtener_columnas_iniciales(archivos[2019][0], archivos[2019][1])
columnas_2025 = obtener_columnas_iniciales(archivos[2025][0], archivos[2025][1])

print("1. COMPARANDO ESTRUCTURA DE NOMBRES:\n")
comparar_columnas_anios(columnas_2019, columnas_2025, 2019, 2025)

1. COMPARANDO ESTRUCTURA DE NOMBRES:

---- Coinciden exactamente (22) ----
  CATEGORIA
  DEPARTAMENTO
  DIRESA
  DISTRITO
  ESTRATEGICOS
  MICRORRED
  PCM
  PEA
  PLIEGO
  PROVINCIA
  Quintil
  RED
  TIPO
  UBIGEO
  UE
  condicion_especialidad
  condicion_laboral
  es_especialista
  especialidad
  id_especialidad
  regimen_laboral
  sexo
---- Solo en 2019 (24) ----
  APS 2015
  CALSIFICACION
  CARGO_ESTRUCTURAL
  CODCARGO
  DESCRIPCION ESTABLECIMIENTO
  DESCRIPCION PLIEGO
  Dist Frontera
  EMERGENCIA
 (*1*)D.S. 136-2019-PCM 
Desde el: 26 Julio  Hasta el: 24 Set 
y (*2*) D.S. 135-2019-PCM 
Desde el: 28 Julio, Hasta el: 25 Set
(*3*) D.S. 137-2019-PCM 
Desde el: 27 Julio, Hasta el: 24 Set
  ESTADO
  Grupo Final
  Grupo Final 2
  INSTITUCION
  MICRORRED PRIORIZADA APS
  PLIEGO + DESCRIP
  REANES FINAL
  UE + DESCRIP UE
  UNIDAD EJECUTORA
  VRAEM 2016 (DS 040-2016-PCM)
  VRAEM 2017 (DS 112-2017-PCM)
  ZAF 2014 FINAL
  cargo
  fecha_nacimiento
  id_cargo
  id_condicion_especialidad
---- Solo

In [ ]:
# 2.2 CARGA REAL DE DATAFRAMES (Obligatorio para poder analizar los valores internos)

print("\nCargando DataFrames completos para auditoría de contenido...")
df_2019 = cargar_dataframe_limpio(2019, archivos)
df_2025 = cargar_dataframe_limpio(2025, archivos)

ANALIZANDO CAMBIOS DE NOMENCLATURA: 2019 VS 2025

---- Coinciden exactamente (22) ----
  CATEGORIA
  DEPARTAMENTO
  DIRESA
  DISTRITO
  ESTRATEGICOS
  MICRORRED
  PCM
  PEA
  PLIEGO
  PROVINCIA
  Quintil
  RED
  TIPO
  UBIGEO
  UE
  condicion_especialidad
  condicion_laboral
  es_especialista
  especialidad
  id_especialidad
  regimen_laboral
  sexo
---- Solo en 2019 (24) ----
  APS 2015
  CALSIFICACION
  CARGO_ESTRUCTURAL
  CODCARGO
  DESCRIPCION ESTABLECIMIENTO
  DESCRIPCION PLIEGO
  Dist Frontera
  EMERGENCIA
 (*1*)D.S. 136-2019-PCM 
Desde el: 26 Julio  Hasta el: 24 Set 
y (*2*) D.S. 135-2019-PCM 
Desde el: 28 Julio, Hasta el: 25 Set
(*3*) D.S. 137-2019-PCM 
Desde el: 27 Julio, Hasta el: 24 Set
  ESTADO
  Grupo Final
  Grupo Final 2
  INSTITUCION
  MICRORRED PRIORIZADA APS
  PLIEGO + DESCRIP
  REANES FINAL
  UE + DESCRIP UE
  UNIDAD EJECUTORA
  VRAEM 2016 (DS 040-2016-PCM)
  VRAEM 2017 (DS 112-2017-PCM)
  ZAF 2014 FINAL
  cargo
  fecha_nacimiento
  id_cargo
  id_condicion_especialid

In [33]:
# ---- Verificando datos de fecha de nacimiento excel 2019 ----

df_2019['fecha_nacimiento'].dtype

<StringDtype(storage='python', na_value=nan)>

In [32]:
# ---- Verificando datos de fecha de nacimiento excel 2019 ----
# df_2019[['fecha_nacimiento']]
df_2019['fecha_nacimiento'].head(10)

0    28/02/1980
1    23/06/1987
2    21/12/1986
3    05/12/1953
4    22/04/1954
5    25/12/1954
6    19/05/1956
7    30/09/1975
8    05/06/1961
9    05/03/1958
Name: fecha_nacimiento, dtype: str

In [29]:
# 2.3 CORRECCIÓN DE COLUMNAS E INYECCIÓN DE EDAD 
# Limpiamos las columnas reales del DF por si acaso quedaran residuos
df_2019.columns = df_2019.columns.str.strip()
df_2025.columns = df_2025.columns.str.strip()

# Convertimos fecha y calculamos EDAD en el DF de 2019
df_2019 = convertir_fecha_nacimiento(df_2019)
df_2019['EDAD'] = calcular_edad(df_2019['fecha_nacimiento_dt'], 2019)

Nulos originales: 726
Nulos despues de convertir: 726
No se pueden convertir (formato distinto): 0


In [31]:
# 2.4 AUDITORÍA DE VALORES (Aquí ya funciona perfectamente porque los DFs tienen datos)
print("\n2. COMPARANDO CONTENIDO REAL DE COLUMNAS SOSPECHOSAS:\n")
df_analisis_cargos = comparar_valores(df_2019, 'id_cargo', '2019', df_2025, 'id_cargo_recod', '2025')

# Mostrar los primeros registros del cruce de datos para validar
df_analisis_cargos.head(20)

# ------- OTRA FORMA --------
#comparar_valores(df_2019, 'id_cargo', '2019', df_2025, 'id_cargo_recod', '2025')



2. COMPARANDO CONTENIDO REAL DE COLUMNAS SOSPECHOSAS:

2019 (id_cargo): 220 valores únicos
2025 (id_cargo_recod): 197 valores únicos
¿Son el mismo conjunto? False


,2019 (id_cargo),2025 (id_cargo_recod)
0,CA001,CA001
1,CA002,CA002
2,CA003,CA003
3,CA004,CA004
4,CA006,CA006
5,CA008,CA007
6,CA009,CA008
7,CA010,CA009
8,CA011,CA010
9,CA012,CA011


In [ ]:
# 2.5 Mapeo de columnas de canonicas y de descarte

mapeo_columnas = {
    2019: {

        'CALSIFICACION': 'CLASIFICACION',     # confirmado: CALSIFICACION (2019) = CLASIFICACION (2025)
        'DESCRIPCION ESTABLECIMIENTO': 'DESCRIPCIONESTABLECIMIENTO', # confirmado: DESCRIPCION ESTABLECIMIENTO (2019) = DESCRIPCIONESTABLECIMIENTO (2025)
        'Dist Frontera'    : 'DistFrontera',  # confirmado: Dist Frontera(2019) =  DistFrontera(2025)
        'Grupo Final'      : 'GrupoFinal2', # confirmado: Grupo Final(2019) = GrupoFinal2(2025)
        'Grupo Final 2'    : 'GrupoFinal3',   # confirmado: Grupo Final(2019) = GrupoFinal3(2025)
        'PLIEGO + DESCRIP' : 'PLIEGODESCRIP', # confirmado: PLIEGO + DESCRIP (2019) = PLIEGODESCRIP (2025)
        'REANES FINAL'     : 'RENAES',        # confirmado: REANES FINAL(2019) = RENAES (2025) *** Debe ser de 8 digitos NO NO NO numerico******
        'UE + DESCRIP UE'  : 'UEDESCRIPUE',   # confirmado: UE + DESCRIP UE(2019) = UEDESCRIPUE(2025)
        'ZAF 2014 FINAL'   : 'ZAF2014FINAL',  # confirmado: ZAF 2014 FINAL(2019) = ZAF2014FINAL(2025)  
        'cargo'            : 'CARGO',         # Solo modificamos a mayuscula por estilo
        'id_cargo'         : 'ID_CARGO'       # Solo modificamos a mayuscula por estilo
    },
    2025: {
        # 2025 ya usa los nombres canónicos para todos estos campos, no necesita entradas
        'cargo_recod'     : 'CARGO',    # confirmado: cargo_recod(2025) = cargo(2019)
        'id_cargo_recod'  : 'ID_CARGO', # confirmado: id_cargo_recod(2025) = id_cargo(2019)
        'Edad'            : 'EDAD'      # Se mantiene porque en excel 2019 se obtuvo EDAD,
    
    },
}

In [ ]:
columnas_descartadas = {
    2019: {
        'APS 2015'                      : 'Solo existe en 2019, sin equivalente en años posteriores',
        'CARGO_ESTRUCTURAL'             : 'No existe en BD 2025 aunque es importante',
        'CODCARGO'                      : 'No es necesario tiene codigo errados',
        'DESCRIPCION PLIEGO'            : 'No necesario porque es lo mismo que PLIEGO + DESCRIP',

        'EMERGENCIA (*1*)D.S. 136-2019-PCM Desde el: 26 Julio Hasta el: 24 Set y (*2*) D.S. 135-2019-PCM Desde el: 28 Julio, Hasta el: 25 Set (*3*) '
        'D.S. 137-2019-PCM Desde el: 27 Julio, Hasta el: 24 Set': 'Campo no necesario porque cada 3 meses se actualiza y no está actualizado',
        
        'ESTADO'                        : 'No necesario, sin equivalente en años posteriores',
        'INSTITUCION'                   : 'No necesario, sin equivalente en años posteriores',
        'MICRORRED PRIORIZADA APS'      : 'No necesario, sin equivalente en años posteriores',
        'VRAEM 2016 (DS 040-2016-PCM)'  : 'No necesario, sin equivalente en años posteriores',
        'VRAEM 2017 (DS 112-2017-PCM)'  : 'No necesario, sin equivalente en años posteriores',
        'id_condicion_especialidad'     : 'No necesario, sin equivalente en años posteriores',
        'fecha_nacimiento'              : 'Transformado a fecha_nacimiento_dt y luego a EDAD; no se conserva en el consolidado final',
        'UNIDAD EJECUTORA'              : 'Duplicado de UE + DESCRIP UE',
    },
    2025: {
        'COMUNIDAD_INDIGENAREFERENCIADGAINPORUBIGEOFEBRERO2022' : 'No necesario, sin equivalente en años anteriores',
        'DESCRIPCIONPLIEGO'                                     : 'Obtenido de PLIEGODESCRIP',
        'DOBLEEMPLEO'                                           : 'No necesario, sin equivalente en años anteriores',
        'EESSCLASJULIO2022'                                     : 'No necesario, sin equivalente en años anteriores',
        'EMERGENCIA1D.S.117123133'                           : 'No necesario, sin equivalente en años anteriores',
        'FRIAJEPORUBIGEO20222024'                               : 'No necesario, sin equivalente en años anteriores',
        'GrupoFinal1'                                           : 'No necesario, se puede obtener de GrupoFinal2',
        'Grupoetareo'                                           : 'No tiene equivalente. Muy necesario, pero se puede obtener de EDAD (contiene grupo por edad)',
        'HELADASPORUBIGEO20222024'                              : 'No necesario, sin equivalente en años anteriores',
        'NIVEL'                                                 : 'No necesario, sin equivalente en años anteriores. Se puede obtener de CATEGORIA',
        'RISAL11NOVIEMBRE2024'                                  : 'No necesario, sin equivalente en años anteriores',
        'VRAEM2022DS1332022PCM'                                 : 'No necesario, sin equivalente en años anteriores',
        'profesion'                                             : 'No necesario, sin equivalente en años anteriores',
        'UNIDADEJECUTORA'                                       : 'Duplicado de UEDESCRIPUE'
    },
}

## **<font color='#eda985'> 3. Fase de Homologación Secuencial (Año a Año) </font>**

## **<font color='#96f499'> 3.1. Comparación: 2019 vs 2020 </font>**

In [ ]:
# =============================================================================
# 3.1. COMPARACIÓN: 2019 VS 2020
# =============================================================================

# 1. Comparación rápida de nombres

columnas_2020 = obtener_columnas_iniciales(archivos[2020][0], archivos[2020][1])
comparar_columnas_anios(columnas_2019, columnas_2020, 2019, 2020)


In [ ]:
# 2. Si detectas una columna sospechosa, cargas el DF real de 2020:

df_2020 = cargar_dataframe_limpio(2020, archivos)
df_2020.columns = df_2020.columns.str.strip()


In [ ]:
# ---- Verificando tipo datos de fecha de nacimiento excel 2019 ----
df_2020['fecha_nacimiento'].dtype

In [ ]:
# ---- Verificando datos de fecha de nacimiento excel 2019 ----
# df_2019[['fecha_nacimiento']]
df_2020['fecha_nacimiento'].head(10)

In [ ]:
# 3. Analizas el contenido de las columnas que te generen dudas. Ejemplo:
# df_analisis_2020 = comparar_valores(df_2019, 'columna_19', '2019', df_2020, 'columna_20', '2020')

In [ ]:
# 4. Inicializar los diccionarios para el año 2020
# (Rellena aquí los campos según lo que imprima la comparación de arriba)
mapeo_columnas[2020] = {
    # 'COLUMNA_2020': 'NOMBRE_CANÓNICO',
}

columnas_descartadas[2020] = {
    # 'COLUMNA_A_DESCARTAR': 'Motivo del descarte',
}

In [ ]:
# 4. Captura dinámica de columnas de EMERGENCIA para 2020 si existiesen
col_emergencia_2020 = [c for c in columnas_por_anio[2020] if str(c).startswith('EMERGENCIA')]
if col_emergencia_2020:
    columnas_descartadas[2020][col_emergencia_2020[0]] = 'Actualizaciones trimestrales no vigentes'

## **<font color='#96f499'> 3.2. Comparación: 2020 vs 2021 </font>**

## **<font color='#96f499'> 3.3. Comparación: 2021 vs 2022 </font>**

## **<font color='#96f499'> 3.4. Comparación: 2022 vs 2023 </font>**

## **<font color='#96f499'> 3.5. Comparación: 2023 vs 2024 </font>**

## **<font color='#96f499'> 3.5. Comparación: 2024 vs 2025 </font>**

#### **Verificando tipo dato y formato de fecha de nacimiento**

<StringDtype(storage='python', na_value=nan)>

*Se observa que la fecha de nacimiento del Excel 2019 tiene 726 registros nuelos*

In [114]:
# ---- Convirtiendo Fecha de nacimiento tipo string a tipo datetime ----
df_2019 = convertir_fecha_nacimiento(df_2019)


Nulos originales: 726
Nulos despues de convertir: 726
No se pueden convertir (formato distinto): 0


*Se observó que existen 73 registros que no se pudieron convertir el formato, es necesario identificarlo*

In [115]:
# Exploracion de registros que no se pudieron convertir
problematicas = df_2019[df_2019['fecha_nacimiento'].notna() & df_2019['fecha_nacimiento_dt'].isna()]
problematicas['fecha_nacimiento'].head(20)

Series([], Name: fecha_nacimiento, dtype: str)

#### **Obteniendo Edad desde la fecha de nacimiento**

In [117]:
df_2019['EDAD'] = calcular_edad(df_2019['fecha_nacimiento_dt'], 2019)

In [118]:
# ---- Verificando el resultado ---
df_2019['EDAD'].head(10)

0    39.0
1    32.0
2    33.0
3    66.0
4    65.0
5    65.0
6    63.0
7    44.0
8    58.0
9    61.0
Name: EDAD, dtype: float64

In [119]:
# ---- Verificando el resultado ---
df_2019['EDAD'].describe()
#df_2019['EDAD'].isna().sum()

count    215728.000000
mean         43.913715
std          12.059142
min          17.000000
25%          34.000000
50%          43.000000
75%          53.000000
max          86.000000
Name: EDAD, dtype: float64

In [120]:
# Verificando los cargos de los registros con edad menores de 20 años
df_2019[df_2019['EDAD'] < 20][['EDAD', 'cargo']].value_counts()

EDAD  cargo                                     
19.0  AUXILIAR ADMINISTRATIVO                       16
      DIGITADOR/A                                   13
      TRABAJADOR/A DE SERVICIOS GENERALES           10
18.0  AUXILIAR ADMINISTRATIVO                        4
      TRABAJADOR/A DE SERVICIOS GENERALES            4
19.0  TECNICO/A EN SEGURIDAD                         2
      TECNICO/A EN SERVICIOS GENERALES I             1
      TECNICO/A EN SOPORTE INFORMATICO               1
      TECNICO/A EN ENFERMERIA I                      1
18.0  TECNICO/A EN SERVICIOS GENERALES I             1
      AUXILIAR DE NUTRICION                          1
17.0  ESPECIALISTA EN SOPORTE INFORMATICO            1
19.0  TECNICO/A EN NUTRICION I                       1
      ENFERMERA/O                                    1
      AUXILIAR DE NUTRICION                          1
      TECNICO/A ADMINISTRATIVO I                     1
      AUXILIAR SANITARIO                             1
      SUPERVISOR

In [121]:
df_2019.head(10)

,PEA,REANES FINAL,INSTITUCION,PLIEGO,DESCRIPCION PLIEGO,PLIEGO + DESCRIP,UE,UNIDAD EJECUTORA,UE + DESCRIP UE,UBIGEO,...,regimen_laboral,condicion_laboral,CODCARGO,CARGO_ESTRUCTURAL,id_cargo,cargo,Grupo Final,Grupo Final 2,fecha_nacimiento_dt,EDAD
0,1,1000000,MINSA,11,M. DE SALUD,011 M. DE SALUD,117,ADMINISTRACION CENTRAL - MINSA,0117 ADMINISTRACION CENTRAL - MINSA,150113,...,Regimen 276,No especifica,76.0,ENF-10,CA168,ENFERMERA/O,Profesional Asistencial,Enfermero,1980-02-28,39.0
1,1,1000000,MINSA,11,M. DE SALUD,011 M. DE SALUD,117,ADMINISTRACION CENTRAL - MINSA,0117 ADMINISTRACION CENTRAL - MINSA,150113,...,Regimen 276,No especifica,NaN,NaN,CA179,TRABAJADOR/A SOCIAL,Profesional Asistencial,Trabajadora Social,1987-06-23,32.0
2,1,1000000,MINSA,11,M. DE SALUD,011 M. DE SALUD,117,ADMINISTRACION CENTRAL - MINSA,0117 ADMINISTRACION CENTRAL - MINSA,150113,...,Regimen 276,No especifica,NaN,NaN,CA168,ENFERMERA/O,Profesional Asistencial,Enfermero,1986-12-21,33.0
3,1,1000000,MINSA,11,M. DE SALUD,011 M. DE SALUD,117,ADMINISTRACION CENTRAL - MINSA,0117 ADMINISTRACION CENTRAL - MINSA,150113,...,Regimen 276,Nombrado,95.0,STB,CNN8,TECNICO ADMINISTRATIVO NO ESPECIFICADO,Tecnico Administrativo,Tecnico Administrativo,1953-12-05,66.0
4,1,1000000,MINSA,11,M. DE SALUD,011 M. DE SALUD,117,ADMINISTRACION CENTRAL - MINSA,0117 ADMINISTRACION CENTRAL - MINSA,150113,...,Regimen 276,Nombrado,101.0,SAB,CA157,AUXILIAR ADMINISTRATIVO,Auxiliar Administrativo,Auxiliar Administrativo,1954-04-22,65.0
5,1,1000000,MINSA,11,M. DE SALUD,011 M. DE SALUD,117,ADMINISTRACION CENTRAL - MINSA,0117 ADMINISTRACION CENTRAL - MINSA,150113,...,Regimen 276,Nombrado,96.0,STC,CA155,TECNICO/A ADMINISTRATIVO II,Tecnico Administrativo,Tecnico Administrativo,1954-12-25,65.0
6,1,1000000,MINSA,11,M. DE SALUD,011 M. DE SALUD,117,ADMINISTRACION CENTRAL - MINSA,0117 ADMINISTRACION CENTRAL - MINSA,150113,...,Regimen 276,Nombrado,94.0,STA,CA156,TECNICO/A ADMINISTRATIVO I,Tecnico Administrativo,Tecnico Administrativo,1956-05-19,63.0
7,1,1000000,MINSA,11,M. DE SALUD,011 M. DE SALUD,117,ADMINISTRACION CENTRAL - MINSA,0117 ADMINISTRACION CENTRAL - MINSA,150113,...,Regimen 276,Nombrado,101.0,SAB,CA157,AUXILIAR ADMINISTRATIVO,Auxiliar Administrativo,Auxiliar Administrativo,1975-09-30,44.0
8,1,1000000,MINSA,11,M. DE SALUD,011 M. DE SALUD,117,ADMINISTRACION CENTRAL - MINSA,0117 ADMINISTRACION CENTRAL - MINSA,150113,...,Regimen 276,Nombrado,101.0,SAB,CA157,AUXILIAR ADMINISTRATIVO,Auxiliar Administrativo,Auxiliar Administrativo,1961-06-05,58.0
9,1,1000000,MINSA,11,M. DE SALUD,011 M. DE SALUD,117,ADMINISTRACION CENTRAL - MINSA,0117 ADMINISTRACION CENTRAL - MINSA,150113,...,Regimen 276,Nombrado,95.0,STB,CNN8,TECNICO ADMINISTRATIVO NO ESPECIFICADO,Tecnico Administrativo,Tecnico Administrativo,1958-03-05,61.0


In [122]:
df_2019.describe()

,PEA,REANES FINAL,PLIEGO,UE,UBIGEO,ESTADO,Quintil,Dist Frontera,VRAEM 2016 (DS 040-2016-PCM),VRAEM 2017 (DS 112-2017-PCM),"EMERGENCIA\n (*1*)D.S. 136-2019-PCM \nDesde el: 26 Julio Hasta el: 24 Set \ny (*2*) D.S. 135-2019-PCM \nDesde el: 28 Julio, Hasta el: 25 Set\n(*3*) D.S. 137-2019-PCM \nDesde el: 27 Julio, Hasta el: 24 Set",ZAF 2014 FINAL,ESTRATEGICOS,MICRORRED PRIORIZADA APS,APS 2015,id_condicion_especialidad,CODCARGO,fecha_nacimiento_dt,EDAD
count,216454.0,2.164540e+05,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,216454.000000,31172.000000,136565.000000,215728,215728.000000
mean,1.0,9.547615e+05,323.753574,1002.552237,128496.122816,0.998425,3.304882,0.051267,0.028514,0.028630,0.028935,0.055074,0.223521,0.173464,0.143924,1.317914,104.582939,1975-08-13 21:50:58.251131040,43.913715
min,1.0,1.000000e+00,11.000000,117.000000,10101.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1933-03-09 00:00:00,17.000000
25%,1.0,2.864000e+03,11.000000,787.000000,80603.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,76.000000,1966-02-27 00:00:00,34.000000
50%,1.0,5.793000e+03,446.000000,1006.000000,150101.000000,1.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,96.000000,1977-01-07 00:00:00,43.000000
75%,1.0,6.723000e+03,454.000000,1322.000000,150201.000000,1.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,103.000000,1985-07-21 00:00:00,53.000000
max,1.0,2.000000e+07,464.000000,1708.000000,250401.000000,1.000000,5.000000,1.000000,2.000000,2.000000,3.000000,1.000000,1.000000,2.000000,2.000000,4.000000,327.000000,2002-03-16 00:00:00,86.000000
std,0.0,3.188331e+06,197.553961,477.622146,58127.192085,0.039660,1.413957,0.220543,0.218606,0.218496,0.251120,0.228125,0.416605,0.475800,0.434761,0.786982,54.927835,NaN,12.059142


In [ ]:
##### AQUI ----

In [159]:
# ---- Como el campo tiene una descripicion muy extensa se generar una variable como key que guarde ese campo ----
'EMERGENCIA (*1*)D.S. 136-2019-PCM Desde el: 26 Julio  Hasta el: 24 Set y (*2*) D.S. 135-2019-PCM Desde el: 28 Julio, '
'Hasta el: 25 Set(*3*) D.S. 137-2019-PCM Desde el: 27 Julio, Hasta el: 24 Set'

col_emergencia_2019 = [c for c in columnas_por_anio[2019] if str(c).startswith('EMERGENCIA')][0]
print(repr(col_emergencia_2019))  # para verlo exacto, con los \n visibles

'EMERGENCIA\n (*1*)D.S. 136-2019-PCM \nDesde el: 26 Julio  Hasta el: 24 Set \ny (*2*) D.S. 135-2019-PCM \nDesde el: 28 Julio, Hasta el: 25 Set\n(*3*) D.S. 137-2019-PCM \nDesde el: 27 Julio, Hasta el: 24 Set'


In [161]:
# ---- Se agregar la variable al diccionario de descarte ---
columnas_descartadas[2019][col_emergencia_2019] = 'Campo no necesario porque cada 3 meses se actualiza y no está actualizado'

In [162]:
# ---- Como el campo tiene una descripicion muy extensa se generar una variable como key que guarde ese campo ----
'EMERGENCIA1D.S.117123133Desdeel20SetiembreHastael25Enero2026Algu'

col_emergencia_2025= [c for c in columnas_por_anio[2025] if str(c).startswith('EMERGENCIA')][0]
print(repr(col_emergencia_2025))  # para verlo exacto, con los \n visibles

'EMERGENCIA1D.S.117123133Desdeel20SetiembreHastael25Enero2026Algu'


In [163]:
# ---- Se agregar la variable al diccionario de descarte ---
columnas_descartadas[2025][col_emergencia_2025] = 'Campo no necesario porque cada 3 meses se actualiza y no está actualizado'

In [164]:
def columnas_que_se_mantienen(columnas_del_anio, anio):
    """Columnas que no requieren rename ni fueron descartadas: se mantienen tal cual."""
    renombradas = set(mapeo_columnas.get(anio, {}).keys())
    descartadas = set(columnas_descartadas.get(anio, {}).keys())
    return sorted(set(columnas_del_anio) - renombradas - descartadas)


In [165]:
columnas_que_se_mantienen(columnas_por_anio[2019], 2019)

['CATEGORIA',
 'DEPARTAMENTO',
 'DIRESA',
 'DISTRITO',
 'ESTRATEGICOS',
 'MICRORRED',
 'PCM',
 'PEA',
 'PLIEGO',
 'PROVINCIA',
 'Quintil',
 'RED',
 'TIPO',
 'UBIGEO',
 'UE',
 'condicion_especialidad',
 'condicion_laboral',
 'es_especialista',
 'especialidad',
 'id_especialidad',
 'regimen_laboral',
 'sexo']

In [167]:
finales_2019 = set(columnas_finales(columnas_por_anio[2019], 2019))
finales_2025 = set(columnas_finales(columnas_por_anio[2025], 2025))

print(f"2019 quedaría con {len(finales_2019)} columnas")
print(f"2025 quedaría con {len(finales_2025)} columnas")
print(f"¿Coinciden exactamente? {finales_2019 == finales_2025}")
print(f"En 2019 pero no en 2025: {finales_2019 - finales_2025}")
print(f"En 2025 pero no en 2019: {finales_2025 - finales_2019}")

2019 quedaría con 33 columnas
2025 quedaría con 34 columnas
¿Coinciden exactamente? False
En 2019 pero no en 2025: set()
En 2025 pero no en 2019: {'EDAD'}
